# Stage 4 - Explore data

What is in the cleaned data before labelling: sizes, lengths, projects, tools, relevance, and topic clusters that sanity-check the domain taxonomy. Outputs go to `results/eda/` (tables as CSV, plots as PNG, `summary.json`).

**Runtime → GPU** speeds up the topic step (it embeds up to 20,000 segments). Settings: `configs/explore.yaml`.

In [ ]:
# --- Colab setup (skip when running locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "https://github.com/WarunaS/knowledge-retention-ml.git"
    !git clone -q {REPO} /content/knowledge-retention-ml || git -C /content/knowledge-retention-ml pull -q
    %pip install -q -e "/content/knowledge-retention-ml[explore]"
    os.environ["KRS_DATA_ROOT"] = "/content/drive/MyDrive/krs/data"
    os.environ["KRS_MODELS_DIR"] = "/content/drive/MyDrive/krs/models"
    os.environ["KRS_RESULTS_DIR"] = "/content/drive/MyDrive/krs/results"

In [ ]:
from krs.explore import run_explore

summary = run_explore("explore")
summary

In [ ]:
import pandas as pd
from krs.config import RESULTS_DIR

eda = RESULTS_DIR / "eda"
pd.read_csv(eda / "overview.csv", index_col=0)   # per-source table for the report

In [ ]:
pd.read_csv(eda / "relevance.csv", index_col=0)

In [ ]:
pd.read_csv(eda / "top_tools.csv")

In [ ]:
from IPython.display import Image, display

for name in summary["plots"]:
    display(Image(filename=str(eda / name)))

## Topics and the domain taxonomy

Each topic gets the domain whose keywords appear most in its segments. A large `other` share in `topic_domain_coverage.csv` means topics the taxonomy has no label for - decide in Stage 4b whether the codebook needs a new domain.

In [ ]:
pd.read_csv(eda / "topics.csv").head(30)

In [ ]:
pd.read_csv(eda / "topic_domain_coverage.csv", index_col=0)

## Share of knowledge-bearing segments

Needs labels, so run this after Stage 4b:

```python
from krs.explore import knowledge_share
from krs.config import get_paths
knowledge_share(get_paths().labels / "weak.jsonl")
```